# 📦 Cell 1: Install Required Packages
Install `psycopg2-binary` and `python-dotenv` inside the Jupyter container.
These are needed to connect to Supabase (PostgreSQL) and load environment variables.

In [13]:

import subprocess
subprocess.run(["pip", "install", "psycopg2-binary", "python-dotenv", "-q"])
print("Done!")

Done!


# ⚡ Cell 2: Initialize Spark Session
Create a SparkSession with the required packages:
- `spark-sql-kafka`: to read streaming data from Kafka
- `postgresql`: JDBC driver to write data to Supabase

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CryptoStreaming") \
    .config("spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0,"
            "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark is ready! ✅")

Spark version: 3.5.0
Spark is ready! ✅


# 🔐 Cell 3: Supabase Connection Configuration
Load credentials from `.env` file and build the JDBC connection URL.
Credentials are never hardcoded — always loaded from environment variables.

In [3]:
import os
from dotenv import load_dotenv

# Load .env
load_dotenv("/home/jovyan/work/.env")

SUPABASE_HOST = os.getenv("SUPABASE_HOST", "aws-0-eu-west-1.pooler.supabase.com")
SUPABASE_PORT = os.getenv("SUPABASE_PORT", "5432")
SUPABASE_DB = os.getenv("SUPABASE_DB", "postgres")
SUPABASE_USER = os.getenv("SUPABASE_USER", "postgres.anfbmwrwjyjqfznqdeer")
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD", "Ms*177912003")

JDBC_URL = f"jdbc:postgresql://{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}"

JDBC_PROPERTIES = {
    "user": SUPABASE_USER,
    "password": SUPABASE_PASSWORD,
    "driver": "org.postgresql.Driver"
}

print("JDBC URL:", JDBC_URL)
print("Connection config ready! ✅")

JDBC URL: jdbc:postgresql://aws-0-eu-west-1.pooler.supabase.com:5432/postgres
Connection config ready! ✅


# 🧪 Cell 4: Test Supabase Connection
Read `dim_currency` table from Supabase to verify the JDBC connection is working correctly.

In [4]:
test_df = spark.read \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "dim_currency") \
    .option("user", SUPABASE_USER) \
    .option("password", SUPABASE_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()

test_df.show()
print("Supabase connection successful! ✅")

+------------+-------------+
|currency_key|currency_code|
+------------+-------------+
|           1|          USD|
|           2|         USDT|
|           3|          EUR|
|          -1|      UNKNOWN|
+------------+-------------+

Supabase connection successful! ✅


# 🥉 Cell 5: Bronze Layer — Raw Data from Kafka
Read raw streaming data from Kafka topic `crypto_market`.
Parse JSON messages using a predefined schema.
This is the raw unprocessed layer.

In [5]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *


schema = StructType([
    StructField("symbol", StringType()),
    StructField("coin_name", StringType()),
    StructField("exchange", StringType()),
    StructField("current_price", DoubleType()),
    StructField("volume_24h", DoubleType()),
    StructField("market_rank", IntegerType()),
    StructField("market_cap", DoubleType()),
    StructField("price_change_24h", DoubleType()),
    StructField("high_24h", DoubleType()),
    StructField("low_24h", DoubleType()),
    StructField("timestamp", StringType()),
    StructField("currency", StringType())
])


bronze_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "crypto_market") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(from_json(col("value").cast("string"), schema).alias("data")) \
    .select("data.*")

print("Bronze Layer ready! ✅")
print("Schema:")
bronze_df.printSchema()

Bronze Layer ready! ✅
Schema:
root
 |-- symbol: string (nullable = true)
 |-- coin_name: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- current_price: double (nullable = true)
 |-- volume_24h: double (nullable = true)
 |-- market_rank: integer (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- currency: string (nullable = true)



# 🥈 Cell 6: Silver Layer — Cleaning & Transformations
Apply the following transformations:
- Uppercase symbol and currency
- Convert timestamp to proper format
- Remove negative prices
- Deduplicate records
- Generate `time_key` in format YYYYMMDDHH
- Add `ingestion_time`

In [6]:
from pyspark.sql.functions import (
    col, upper, trim, when, to_timestamp,
    hour, dayofmonth, month, quarter, year,
    dayofweek, date_format, current_timestamp
)

silver_df = bronze_df \
    .withColumn("symbol", upper(trim(col("symbol")))) \
    .withColumn("currency", upper(trim(col("currency")))) \
    .withColumn("exchange", trim(col("exchange"))) \
    .withColumn("coin_name", trim(col("coin_name"))) \
    .withColumn("timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("current_price", when(col("current_price") < 0, None).otherwise(col("current_price"))) \
    .withColumn("volume_24h", when(col("volume_24h") < 0, None).otherwise(col("volume_24h"))) \
    .withColumn("market_cap", when(col("market_cap") < 0, None).otherwise(col("market_cap"))) \
    .withColumn("time_key", 
        (year(col("timestamp")) * 1000000 +
         month(col("timestamp")) * 10000 +
         dayofmonth(col("timestamp")) * 100 +
         hour(col("timestamp"))).cast("long")) \
    .withColumn("ingestion_time", current_timestamp()) \
    .dropDuplicates(["symbol", "exchange", "timestamp"]) \
    .filter(col("current_price").isNotNull())

print("Silver Layer ready! ✅")
print("Schema:")
silver_df.printSchema()

Silver Layer ready! ✅
Schema:
root
 |-- symbol: string (nullable = true)
 |-- coin_name: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- current_price: double (nullable = true)
 |-- volume_24h: double (nullable = true)
 |-- market_rank: integer (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- time_key: long (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



# 📊 Cell 7: Load Dimension Tables from Supabase
Load all dimension tables needed for the Star Schema lookups:
- `dim_coin`
- `dim_exchange`
- `dim_currency`

In [7]:

dim_coin_df = spark.read \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "dim_coin") \
    .option("user", SUPABASE_USER) \
    .option("password", SUPABASE_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()

dim_exchange_df = spark.read \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "dim_exchange") \
    .option("user", SUPABASE_USER) \
    .option("password", SUPABASE_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()

dim_currency_df = spark.read \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "dim_currency") \
    .option("user", SUPABASE_USER) \
    .option("password", SUPABASE_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()

print("Dimensions loaded! ✅")
dim_coin_df.show(5)
dim_exchange_df.show()
dim_currency_df.show()

Dimensions loaded! ✅
+--------+------+------------+---------------+
|coin_key|symbol|   coin_name|       category|
+--------+------+------------+---------------+
|       1|   BTC|     Bitcoin| Store of Value|
|       2|   ETH|    Ethereum|Smart Contracts|
|       3|   SOL|      Solana|        Layer 1|
|       4|   BNB|Binance Coin| Exchange Token|
|       5|   ADA|     Cardano|        Layer 1|
+--------+------+------------+---------------+
only showing top 5 rows

+------------+-------------+-------+
|exchange_key|exchange_name|country|
+------------+-------------+-------+
|           1|    CoinGecko| Global|
|           2|      Binance|    UAE|
|           3|     Coinbase|    USA|
|           4|       Kraken|    USA|
|          -1|      UNKNOWN|Unknown|
+------------+-------------+-------+

+------------+-------------+
|currency_key|currency_code|
+------------+-------------+
|           1|          USD|
|           2|         USDT|
|           3|          EUR|
|          -1|      UNK

# ✍️ Cell 8: Write to Supabase — Gold Layer
For each micro-batch:
1. Join Silver data with Dimension tables to get surrogate keys
2. UPSERT new coins into `dim_coin`
3. Insert enriched fact records into `fact_crypto_market_snapshot`

In [8]:
import psycopg2

def write_to_supabase(batch_df, batch_id):
    print(f"Processing batch {batch_id}...")
    
   
    gold_df = batch_df \
        .join(dim_coin_df.select("coin_key", "symbol"), on="symbol", how="left") \
        .join(dim_exchange_df.select("exchange_key", "exchange_name"), 
              batch_df.exchange == dim_exchange_df.exchange_name, how="left") \
        .join(dim_currency_df.select("currency_key", "currency_code"),
              batch_df.currency == dim_currency_df.currency_code, how="left") \
        .fillna({"coin_key": -1, "exchange_key": -1, "currency_key": -1}) \
        .select(
            col("coin_key").cast("int"),
            col("exchange_key").cast("int"),
            col("time_key").cast("long"),
            col("currency_key").cast("int"),
            col("current_price").alias("price_usd"),
            col("volume_24h"),
            col("market_rank"),
            col("market_cap"),
            col("price_change_24h"),
            col("high_24h"),
            col("low_24h"),
            col("ingestion_time")
        )

    
    conn = psycopg2.connect(
        host=SUPABASE_HOST,
        port=SUPABASE_PORT,
        dbname=SUPABASE_DB,
        user=SUPABASE_USER,
        password=SUPABASE_PASSWORD
    )
    cursor = conn.cursor()

    for row in batch_df.select("symbol", "coin_name").distinct().collect():
        cursor.execute("""
            INSERT INTO dim_coin (symbol, coin_name, category)
            VALUES (%s, %s, %s)
            ON CONFLICT (symbol) DO NOTHING
        """, (row.symbol, row.coin_name, "Cryptocurrency"))
    
    conn.commit()
    cursor.close()
    conn.close()

    # كتابة الـ Fact
    gold_df.write \
        .format("jdbc") \
        .option("url", JDBC_URL) \
        .option("dbtable", "fact_crypto_market_snapshot") \
        .option("user", SUPABASE_USER) \
        .option("password", SUPABASE_PASSWORD) \
        .option("driver", "org.postgresql.Driver") \
        .mode("append") \
        .save()

    print(f"Batch {batch_id} written successfully! ✅")

print("Write function ready! ✅")

Write function ready! ✅


# 🚀 Cell 9: Start Streaming Query
Start the PySpark Structured Streaming pipeline.
- Trigger every 60 seconds
- Uses `foreachBatch` to write each micro-batch to Supabase
- Checkpoint location: `/home/jovyan/work/checkpoint`

In [11]:
query = silver_df.writeStream \
    .foreachBatch(write_to_supabase) \
    .outputMode("update") \
    .option("checkpointLocation", "/home/jovyan/work/checkpoint") \
    .trigger(processingTime="60 seconds") \
    .start()

print("Streaming started! ✅")
print("Waiting for data from Kafka...")
query.awaitTermination()

Streaming started! ✅
Waiting for data from Kafka...
Processing batch 8...
Batch 8 written successfully! ✅
Processing batch 9...
Batch 9 written successfully! ✅
Processing batch 10...
Batch 10 written successfully! ✅
Processing batch 11...
Batch 11 written successfully! ✅
Processing batch 12...
Batch 12 written successfully! ✅
Processing batch 13...
Batch 13 written successfully! ✅


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

# 🛑 Cell 10: Stop Streaming Query
Gracefully stop the streaming query when done.

In [ ]:
query.stop()
print("Streaming stopped! ✅")

Processing batch 14...
Batch 14 written successfully! ✅
Processing batch 15...
Batch 15 written successfully! ✅
Processing batch 16...
